In [1]:
from pathlib import Path

input_interval = 5
detect_interval = 300
total_duration = 30
total_demand = 300

main_dir = Path(".")
true_OD_dir = main_dir / "true_OD.pkl"

origin_list = ["N1", "N4"]
destination_list = ["N2", "N3"]
num_OD = len(origin_list) * len(destination_list)

In [3]:
import numpy as np

def make_blockwise_inputs_binary(
    total_duration_min,
    input_interval,
    detect_interval,
    num_OD,
    total_demand,
    block_jitter_frac=0.05,
    base_pull_frac=0.2,
    base_p=None,
    pulse=1,
    seed=None
):
    rng = np.random.default_rng(seed)

    if pulse <= 0:
        raise ValueError("pulse must be a positive integer.")

    if total_demand % pulse != 0:
        raise ValueError(
            f"Under the 0/pulse constraint, total_demand must be a multiple of pulse. "
            f"(total_demand={total_demand}, pulse={pulse})"
        )

    total_seconds = total_duration_min * 60
    num_step = total_seconds // input_interval
    steps_per_block = detect_interval // input_interval
    num_blocks = total_seconds // detect_interval

    if num_step != num_blocks * steps_per_block:
        raise ValueError("duration, detect_interval, and input_interval must divide into integer steps.")

    total_on = total_demand // pulse

    total_elements = num_step * num_OD
    if total_on > total_elements:
        raise ValueError("total_demand/pulse must be less than or equal to num_step * num_OD under the 0/pulse constraint.")

    base = total_on // num_blocks
    rem = total_on % num_blocks
    block_totals = np.full(num_blocks, base, dtype=int)
    block_totals[:rem] += 1

    if base_p is None:
        base_p = np.ones(num_OD, dtype=float) / float(num_OD)
    else:
        base_p = np.asarray(base_p, dtype=float)
        if base_p.shape != (num_OD,):
            raise ValueError("base_p must be a one-dimensional vector of length num_OD.")
        s = float(base_p.sum())
        if s <= 0:
            raise ValueError("The sum of base_p must be positive.")
        if np.any(base_p < 0):
            raise ValueError("base_p cannot contain negative values.")
        base_p = base_p / s

    cap = int(steps_per_block)

    inputs01 = np.zeros((num_step, num_OD), dtype=np.int8)

    c_prev = rng.multinomial(int(block_totals[0]), base_p)

    for b in range(num_blocks):
        Nb = int(block_totals[b])

        if b == 0:
            c_b = c_prev.copy()
        else:
            c_b = c_prev.copy()

            moves = int(round(Nb * block_jitter_frac))
            for _ in range(moves):
                src = rng.integers(num_OD)
                dst = rng.integers(num_OD)
                if src != dst and c_b[src] > 0:
                    c_b[src] -= 1
                    c_b[dst] += 1

            pull_moves = int(round(Nb * base_pull_frac))
            if pull_moves > 0:
                base_counts = np.floor(base_p * Nb).astype(int)
                diff_bc = Nb - int(base_counts.sum())
                if diff_bc > 0:
                    add_idx = rng.choice(num_OD, size=diff_bc, replace=True, p=base_p)
                    np.add.at(base_counts, add_idx, 1)

                for _ in range(pull_moves):
                    over = np.where(c_b > base_counts)[0]
                    under = np.where(c_b < base_counts)[0]
                    if over.size == 0 or under.size == 0:
                        break
                    src = rng.choice(over)
                    dst = rng.choice(under)
                    if src != dst and c_b[src] > 0:
                        c_b[src] -= 1
                        c_b[dst] += 1

            diff = Nb - int(c_b.sum())
            if diff > 0:
                add = rng.choice(num_OD, size=diff, replace=True, p=base_p)
                np.add.at(c_b, add, 1)
            elif diff < 0:
                for _ in range(-diff):
                    pos = np.where(c_b > 0)[0]
                    if pos.size == 0:
                        break
                    k = rng.choice(pos)
                    c_b[k] -= 1

            overflow = np.where(c_b > cap)[0]
            if overflow.size > 0:
                excess = int((c_b[overflow] - cap).sum())
                c_b[overflow] = cap

                free = (cap - c_b).astype(int)
                free_sum = int(free.sum())
                if excess > 0:
                    if free_sum <= 0:
                        raise ValueError(
                            "No capacity is available for reassignment under the 0/pulse constraint. "
                            "Reduce total_demand, increase num_OD, or adjust detect_interval/input_interval."
                        )
                    add = rng.multinomial(excess, free / free_sum)
                    c_b = c_b + add

        start = b * steps_per_block
        for k in range(num_OD):
            ck = int(c_b[k])
            if ck <= 0:
                continue
            if ck > steps_per_block:
                raise ValueError(
                    f"0/pulse constraint violation: block {b}, OD {k} has ck={ck} greater than steps_per_block={steps_per_block}."
                )
            ts_local = rng.choice(steps_per_block, size=ck, replace=False)
            inputs01[start + ts_local, k] = 1

        c_prev = c_b

    if int(inputs01.sum()) != int(total_on):
        raise RuntimeError(f"Sum mismatch: inputs01.sum()={inputs01.sum()} vs total_on={total_on}")

    return (inputs01.astype(np.int64) * int(pulse)).astype(np.int64)

In [4]:
import pickle

inputs = make_blockwise_inputs_binary(
    total_duration_min=total_duration,
    input_interval=input_interval,
    detect_interval=detect_interval,
    num_OD=num_OD,
    total_demand=total_demand,
    block_jitter_frac=0.2,
    base_pull_frac=0.0,
    base_p=None,
    pulse=1,
    seed=101
)

print("inputs shape:", inputs.shape)
print("total demand check:", inputs.sum())

with open(true_OD_dir, "wb") as f:
    pickle.dump(inputs, f)

print("saved:", true_OD_dir)

inputs shape: (360, 4)
total demand check: 300
saved: true_OD.pkl


In [5]:
import pickle
import numpy as np

with open(true_OD_dir, "rb") as f:
    inputs = pickle.load(f)

inputs = np.asarray(inputs)

print("loading OD matrices complete")
print("shape:", inputs.shape)
print("dtype:", inputs.dtype)
print("sum:", inputs.sum())

loading OD matrices complete
shape: (360, 4)
dtype: int64
sum: 300
